In [1]:
!pip install -q --no-index --find-links /kaggle/input/ultralytics ultralytics

In [2]:
%%writefile convert_img.py 

import os
import pydicom
from PIL import Image
import numpy as np
from multiprocessing import Pool, cpu_count

import sklearn.metrics
import torch
import cv2
import numpy as np 
import pandas as pd 
from tqdm.auto import tqdm
IMG_DIR = '/images'
def read_dcm(src_path):
    dicom_data = pydicom.dcmread(src_path)
    image = dicom_data.pixel_array
    image = (image - image.min()) / (image.max() - image.min() +1e-6) * 255
    return image

def convert_dcm_to_jpg(file_path):
    try:
        # Read the DICOM file
        image_array = read_dcm(file_path)
        
        # Define the output path
        relative_path = os.path.relpath(file_path, start=input_directory)
        output_path = os.path.join(output_directory, relative_path)
        output_path = output_path.replace('.dcm', '.jpg')
                
        # Create the output directory if it doesn't exist
        os.makedirs(os.path.dirname(output_path), exist_ok=True)
        
        # Save the image as a JPEG file
        cv2.imwrite(output_path, image_array)
        
        return output_path
    except Exception as e:
        print(f"Error processing file {file_path}: {e}")
        return None

def process_files(dcm_files):
    with Pool(cpu_count()) as pool:
        # Wrap pool.map with tqdm to show the progress bar
        list(tqdm(pool.imap(convert_dcm_to_jpg, dcm_files), total=len(dcm_files)))

def get_dcm_files(directory):
    dcm_files = []
    for root, dirs, files in os.walk(directory):
        for file in files:
            if file.endswith('.dcm'):
                dcm_files.append(os.path.join(root, file))
    return dcm_files    


input_directory = '/kaggle/input/rsna-2024-lumbar-spine-degenerative-classification/test_images'

output_directory = IMG_DIR

# Get all .dcm files in the input directory
dcm_files = get_dcm_files(input_directory)

# Process the files using multiprocessing
process_files(dcm_files)

print(f"Conversion completed. Images saved to {output_directory}")

Writing convert_img.py


In [3]:
!python convert_img.py 

100%|███████████████████████████████████████████| 97/97 [00:00<00:00, 99.60it/s]
Conversion completed. Images saved to /images


In [4]:
%%writefile infer.py 

import sys
model_dir = sys.argv[1]
img_size = int(sys.argv[2])
fold_index = int(sys.argv[3])
save_file = sys.argv[4]
#weight = sys.argv[5]
print(model_dir, img_size, fold_index)

import os
import pydicom
from PIL import Image
import numpy as np
from multiprocessing import Pool, cpu_count

import sklearn.metrics
import torch
import cv2
import numpy as np 
import pandas as pd 
from tqdm.auto import tqdm

EVAL = 0 
IMG_DIR = '/images'
FOLD = int(fold_index)
SAMPLE = 0 
SEVERITIES = ['Normal/Mild', 'Moderate', 'Severe']
LEVELS = ['l1_l2', 'l2_l3', 'l3_l4', 'l4_l5', 'l5_s1']

SCS_WEIGHTS = [f'{model_dir}/scs_fold{FOLD}_{img_size}_normal.pt']
SS_WEIGHTS = [f'{model_dir}/ss_fold{FOLD}_{img_size}_normal.pt']
NFN_WEIGHTS = [f'{model_dir}/nfn_fold{FOLD}_{img_size}_normal.pt']

SCS_WEIGHTS += [f'{model_dir}/scs_fold{FOLD}_{img_size}_severe.pt']
SS_WEIGHTS += [f'{model_dir}/ss_fold{FOLD}_{img_size}_severe.pt']
NFN_WEIGHTS += [f'{model_dir}/nfn_fold{FOLD}_{img_size}_severe.pt']

print(SCS_WEIGHTS,'\n', SS_WEIGHTS,'\n', NFN_WEIGHTS)

train_val_df = pd.read_csv('/kaggle/input/rsna-2024-lumbar-spine-degenerative-classification/train.csv')
des = pd.read_csv('/kaggle/input/rsna-2024-lumbar-spine-degenerative-classification/test_series_descriptions.csv')


test_df = os.listdir('/kaggle/input/rsna-2024-lumbar-spine-degenerative-classification/test_images')
test_df = pd.DataFrame(test_df, columns=['study_id'])
test_df['study_id'] = test_df['study_id'].astype(int)
    
test_df = test_df.merge(des, on=['study_id'])

def gen_label_map(CONDITIONS):
    label2id = {}
    id2label = {}
    i = 0
    for cond in CONDITIONS:
        for level in LEVELS:
            for severity in SEVERITIES:
                cls_ = f"{cond.lower().replace(' ', '_')}_{level}_{severity.lower()}"
                label2id[cls_] = i
                id2label[i] = cls_
                i+=1
    return label2id, id2label
                
scs_label2id, scs_id2label = gen_label_map(['Spinal Canal Stenosis'])
ss_label2id, ss_id2label = gen_label_map(['Left Subarticular Stenosis', 'Right Subarticular Stenosis'])
nfn_label2id, nfn_id2label = gen_label_map(['Left Neural Foraminal Narrowing', 'Right Neural Foraminal Narrowing'])

from ultralytics import YOLO

# Load YOLO Model
scs_models = []
for weight in SCS_WEIGHTS:
    scs_models.append(YOLO(weight))

ss_models = []
for weight in SS_WEIGHTS:
    ss_models.append(YOLO(weight))

nfn_models = []
for weight in NFN_WEIGHTS:
    nfn_models.append(YOLO(weight))

all_label_set = train_val_df.iloc[0, 1:].index.tolist()
scs_label_set = all_label_set[:5]
nfn_label_set = all_label_set[5:15]
ss_label_set = all_label_set[15:]

settings = [
    ( 'Sagittal T2/STIR', scs_models, scs_id2label, scs_label_set, 0.01),
    ( 'Axial T2', ss_models, ss_id2label, ss_label_set, 0.02),
    ( 'Sagittal T1', nfn_models, nfn_id2label, nfn_label_set, 0.02)
]

from collections import defaultdict

pred_rows = []

for modality, models, id2label, label_set, thresh in settings:
    mod_df = test_df[test_df.series_description == modality]
    
    if SAMPLE:
        mod_df = mod_df.sample(20, random_state=610)
    
    # for each study, at each level and condition, get the maximum probability score
    for study_id, group in tqdm(mod_df.groupby('study_id')):
        predictions = defaultdict(list)
        for i, row in group.iterrows():
            # predict on all images from all the series
            series_dir = os.path.join(IMG_DIR, str(row['study_id']), str(row['series_id']))
            for model in models:
                results = model(series_dir, imgsz=img_size, conf=thresh, verbose=False, augment=False, )
                for res in results:
                    for pred_class, conf in zip(res.boxes.cls, res.boxes.conf):
                        pred_class = pred_class.item()
                        conf = conf.item()
                        _class = id2label[pred_class]
                        predictions[_class].append(conf)
        
        # aggregate the result on images to obtain study-level prediction
        for condition in label_set:
            res_dict = {'row_id': f'{study_id}_{condition}' }

            score_vec = []
            for severity in SEVERITIES:
                if severity=='Severe':
                    scaler = 2.1415
                elif severity=='Moderate':
                    scaler = 1.2
                else:
                    scaler = 0.85
                severity = severity.lower()
                key = f'{condition}_{severity}'
                if len(predictions[key]) > 0:
                    score = np.max(predictions[key])
                else:
                    score = thresh
                    
                if score>0.2:
                    score = score*scaler
                    
                score_vec.append(score)
                
            # normalize score to sum to 1
            score_vec = torch.tensor(score_vec)
            score_vec = score_vec / score_vec.sum()

            for idx, severity in enumerate(SEVERITIES):
                res_dict[severity.replace('/', '_').lower()] = score_vec[idx].item()

            pred_rows.append(res_dict)


pred_df = pd.DataFrame(pred_rows)
pred_df

pred_df.to_csv(save_file, index=False)

Writing infer.py


In [5]:
!python infer.py /kaggle/input/yolo-folds-model3/yolov8s_severe_last 384 0 pred1.csv

/kaggle/input/yolo-folds-model3/yolov8s_severe_last 384 0
['/kaggle/input/yolo-folds-model3/yolov8s_severe_last/scs_fold0_384_normal.pt', '/kaggle/input/yolo-folds-model3/yolov8s_severe_last/scs_fold0_384_severe.pt'] 
 ['/kaggle/input/yolo-folds-model3/yolov8s_severe_last/ss_fold0_384_normal.pt', '/kaggle/input/yolo-folds-model3/yolov8s_severe_last/ss_fold0_384_severe.pt'] 
 ['/kaggle/input/yolo-folds-model3/yolov8s_severe_last/nfn_fold0_384_normal.pt', '/kaggle/input/yolo-folds-model3/yolov8s_severe_last/nfn_fold0_384_severe.pt']
100%|█████████████████████████████████████████████| 1/1 [00:00<00:00,  1.10it/s]


In [6]:
!python infer.py /kaggle/input/yolo-folds-model3/yolov8s_severe_last 512 0 pred2.csv

/kaggle/input/yolo-folds-model3/yolov8s_severe_last 512 0
['/kaggle/input/yolo-folds-model3/yolov8s_severe_last/scs_fold0_512_normal.pt', '/kaggle/input/yolo-folds-model3/yolov8s_severe_last/scs_fold0_512_severe.pt'] 
 ['/kaggle/input/yolo-folds-model3/yolov8s_severe_last/ss_fold0_512_normal.pt', '/kaggle/input/yolo-folds-model3/yolov8s_severe_last/ss_fold0_512_severe.pt'] 
 ['/kaggle/input/yolo-folds-model3/yolov8s_severe_last/nfn_fold0_512_normal.pt', '/kaggle/input/yolo-folds-model3/yolov8s_severe_last/nfn_fold0_512_severe.pt']
100%|█████████████████████████████████████████████| 1/1 [00:00<00:00,  1.06it/s]


In [7]:
!python infer.py /kaggle/input/yolo-folds-model3/yolov8s_severe_last 768 0 pred3.csv

/kaggle/input/yolo-folds-model3/yolov8s_severe_last 768 0
['/kaggle/input/yolo-folds-model3/yolov8s_severe_last/scs_fold0_768_normal.pt', '/kaggle/input/yolo-folds-model3/yolov8s_severe_last/scs_fold0_768_severe.pt'] 
 ['/kaggle/input/yolo-folds-model3/yolov8s_severe_last/ss_fold0_768_normal.pt', '/kaggle/input/yolo-folds-model3/yolov8s_severe_last/ss_fold0_768_severe.pt'] 
 ['/kaggle/input/yolo-folds-model3/yolov8s_severe_last/nfn_fold0_768_normal.pt', '/kaggle/input/yolo-folds-model3/yolov8s_severe_last/nfn_fold0_768_severe.pt']
100%|█████████████████████████████████████████████| 1/1 [00:01<00:00,  1.20s/it]


In [8]:
!python infer.py /kaggle/input/yolo-folds-model3/yolov8s_severe_last 768 1 pred4.csv

/kaggle/input/yolo-folds-model3/yolov8s_severe_last 768 1
['/kaggle/input/yolo-folds-model3/yolov8s_severe_last/scs_fold1_768_normal.pt', '/kaggle/input/yolo-folds-model3/yolov8s_severe_last/scs_fold1_768_severe.pt'] 
 ['/kaggle/input/yolo-folds-model3/yolov8s_severe_last/ss_fold1_768_normal.pt', '/kaggle/input/yolo-folds-model3/yolov8s_severe_last/ss_fold1_768_severe.pt'] 
 ['/kaggle/input/yolo-folds-model3/yolov8s_severe_last/nfn_fold1_768_normal.pt', '/kaggle/input/yolo-folds-model3/yolov8s_severe_last/nfn_fold1_768_severe.pt']
100%|█████████████████████████████████████████████| 1/1 [00:01<00:00,  1.14s/it]


In [9]:
!python infer.py /kaggle/input/yolo-folds-model3/yolov8s_severe_last 512 1 pred5.csv

/kaggle/input/yolo-folds-model3/yolov8s_severe_last 512 1
['/kaggle/input/yolo-folds-model3/yolov8s_severe_last/scs_fold1_512_normal.pt', '/kaggle/input/yolo-folds-model3/yolov8s_severe_last/scs_fold1_512_severe.pt'] 
 ['/kaggle/input/yolo-folds-model3/yolov8s_severe_last/ss_fold1_512_normal.pt', '/kaggle/input/yolo-folds-model3/yolov8s_severe_last/ss_fold1_512_severe.pt'] 
 ['/kaggle/input/yolo-folds-model3/yolov8s_severe_last/nfn_fold1_512_normal.pt', '/kaggle/input/yolo-folds-model3/yolov8s_severe_last/nfn_fold1_512_severe.pt']
100%|█████████████████████████████████████████████| 1/1 [00:00<00:00,  1.07it/s]


In [10]:
!python infer.py /kaggle/input/yolo-folds-model3/yolov8s_severe_last 384 1 pred6.csv

/kaggle/input/yolo-folds-model3/yolov8s_severe_last 384 1
['/kaggle/input/yolo-folds-model3/yolov8s_severe_last/scs_fold1_384_normal.pt', '/kaggle/input/yolo-folds-model3/yolov8s_severe_last/scs_fold1_384_severe.pt'] 
 ['/kaggle/input/yolo-folds-model3/yolov8s_severe_last/ss_fold1_384_normal.pt', '/kaggle/input/yolo-folds-model3/yolov8s_severe_last/ss_fold1_384_severe.pt'] 
 ['/kaggle/input/yolo-folds-model3/yolov8s_severe_last/nfn_fold1_384_normal.pt', '/kaggle/input/yolo-folds-model3/yolov8s_severe_last/nfn_fold1_384_severe.pt']
100%|█████████████████████████████████████████████| 1/1 [00:00<00:00,  1.10it/s]


In [11]:
!python infer.py /kaggle/input/yolo-folds-model4/yolov8m_last 384 0 pred7.csv

/kaggle/input/yolo-folds-model4/yolov8m_last 384 0
['/kaggle/input/yolo-folds-model4/yolov8m_last/scs_fold0_384_normal.pt', '/kaggle/input/yolo-folds-model4/yolov8m_last/scs_fold0_384_severe.pt'] 
 ['/kaggle/input/yolo-folds-model4/yolov8m_last/ss_fold0_384_normal.pt', '/kaggle/input/yolo-folds-model4/yolov8m_last/ss_fold0_384_severe.pt'] 
 ['/kaggle/input/yolo-folds-model4/yolov8m_last/nfn_fold0_384_normal.pt', '/kaggle/input/yolo-folds-model4/yolov8m_last/nfn_fold0_384_severe.pt']
100%|█████████████████████████████████████████████| 1/1 [00:01<00:00,  1.30s/it]


In [12]:
!python infer.py /kaggle/input/yolo-folds-model4/yolov8m_last 384 1 pred8.csv

/kaggle/input/yolo-folds-model4/yolov8m_last 384 1
['/kaggle/input/yolo-folds-model4/yolov8m_last/scs_fold1_384_normal.pt', '/kaggle/input/yolo-folds-model4/yolov8m_last/scs_fold1_384_severe.pt'] 
 ['/kaggle/input/yolo-folds-model4/yolov8m_last/ss_fold1_384_normal.pt', '/kaggle/input/yolo-folds-model4/yolov8m_last/ss_fold1_384_severe.pt'] 
 ['/kaggle/input/yolo-folds-model4/yolov8m_last/nfn_fold1_384_normal.pt', '/kaggle/input/yolo-folds-model4/yolov8m_last/nfn_fold1_384_severe.pt']
100%|█████████████████████████████████████████████| 1/1 [00:01<00:00,  1.31s/it]


In [13]:
%%writefile maxvit3d.py
data_path = "/kaggle/input/rsna-2024-lumbar-spine-degenerative-classification/"

img_size = (256, 256)

import pandas as pd

def retrieve_test_data(data_path):
    test_df = pd.read_csv(data_path + 'test_series_descriptions.csv')

    return test_df

retrieve_test_data(data_path)

import os

def retrieve_image_paths(base_path, study_id, series_id):
    series_dir = os.path.join(base_path, str(study_id), str(series_id))
    images = os.listdir(series_dir)
    image_paths = [os.path.join(series_dir, img) for img in images]
    return image_paths

import open3d as o3d
from pydicom import dcmread
import math
import numpy as np
import cv2
import copy

def read_study_as_pcd(dir_path, series_types_dict=None, downsampling_factor=1, img_size=img_size):
    pcd_overall = o3d.geometry.PointCloud()

    for path in glob.glob(os.path.join(dir_path, "**/*.dcm"), recursive=True):
        dicom_slice = dcmread(path)

        series_id = os.path.basename(os.path.dirname(path))
        study_id = os.path.basename(os.path.dirname(os.path.dirname(path)))
        if series_types_dict is None or int(series_id) not in series_types_dict:
            series_desc = dicom_slice.SeriesDescription
        else:
            series_desc = series_types_dict[int(series_id)]
            series_desc = series_desc.split(" ")[-1]

        x_orig, y_orig = dicom_slice.pixel_array.shape
        img = np.expand_dims(cv2.resize(dicom_slice.pixel_array, img_size, interpolation=cv2.INTER_AREA), -1)
        x, y, z = np.where(img)

        downsampling_factor_iter = max(downsampling_factor, int(math.ceil(len(x) / 6e6)))

        index_voxel = np.vstack((x, y, z))[:, ::downsampling_factor_iter]
        grid_index_array = index_voxel.T
        pcd = o3d.geometry.PointCloud(o3d.utility.Vector3dVector(grid_index_array.astype(np.float64)))

        vals = np.expand_dims(img[x, y, z][::downsampling_factor_iter], -1)
        if series_desc == "T1":
            vals = np.pad(vals, ((0, 0), (0, 2)))
        elif series_desc == "T2":
            vals = np.pad(vals, ((0, 0), (1, 1)))
        elif series_desc == "T2/STIR":
            vals = np.pad(vals, ((0, 0), (2, 0)))
        else:
            raise ValueError(f"Unknown series desc: {series_desc}")

        pcd.colors = o3d.utility.Vector3dVector(vals.astype(np.float64))

        dX, dY = dicom_slice.PixelSpacing
        dZ = dicom_slice.SliceThickness

        X = np.array(list(dicom_slice.ImageOrientationPatient[:3]) + [0]) * dX
        Y = np.array(list(dicom_slice.ImageOrientationPatient[3:]) + [0]) * dY

        for z in range(int(dZ)):
            pos = list(dicom_slice.ImagePositionPatient)
            if series_desc == "T2":
                pos[-1] += z
            else:
                pos[0] += z
            S = np.array(pos + [1])

            transform_matrix = np.array([X, Y, np.zeros(len(X)), S]).T
            transform_matrix = transform_matrix @ np.matrix(
                [[0, y_orig / img_size[1], 0, 0],
                 [x_orig / img_size[0], 0, 0, 0],
                 [0, 0, 1, 0],
                 [0, 0, 0, 1]]
            )

            pcd_overall += copy.deepcopy(pcd).transform(transform_matrix)

    return pcd_overall



def read_study_as_voxel_grid(dir_path, series_type_dict=None, downsampling_factor=1, img_size=img_size):
    pcd_overall = read_study_as_pcd(dir_path,
                                    series_types_dict=series_type_dict,
                                    downsampling_factor=downsampling_factor,
                                    img_size=img_size)
    box = pcd_overall.get_axis_aligned_bounding_box()

    max_b = np.array(box.get_max_bound())
    min_b = np.array(box.get_min_bound())

    pts = (np.array(pcd_overall.points) - (min_b)) * (
                (img_size[0] - 1, img_size[0] - 1, img_size[0] - 1) / (max_b - min_b))
    coords = np.round(pts).astype(np.int32)
    vals = np.array(pcd_overall.colors, dtype=np.float16)

    grid = np.zeros((3, img_size[0], img_size[0], img_size[0]), dtype=np.float16)
    indices = coords[:, 0], coords[:, 1], coords[:, 2]

    np.maximum.at(grid[0], indices, vals[:, 0])
    np.maximum.at(grid[1], indices, vals[:, 1])
    np.maximum.at(grid[2], indices, vals[:, 2])


    return grid

import numpy as np
from torch.utils.data import Dataset, DataLoader
import torchio as tio
import torch.nn as nn
import pydicom

CONDITIONS = {
    "Sagittal T2/STIR": ["Spinal Canal Stenosis"],
    "Axial T2": ["Left Subarticular Stenosis", "Right Subarticular Stenosis"],
    "Sagittal T1": ["Left Neural Foraminal Narrowing", "Right Neural Foraminal Narrowing"],
}


class PatientLevelTestset(Dataset):
    def __init__(self,
                 base_path: str,
                 dataframe: pd.DataFrame,
                 transform_3d=None):
        self.base_path = base_path

        self.dataframe = (dataframe[['study_id', "series_id", "series_description"]]
                          .drop_duplicates())

        self.subjects = self.dataframe[['study_id']].drop_duplicates().reset_index(drop=True)
        self.series_descs = {e[0]: e[1] for e in self.dataframe[["series_id", "series_description"]].drop_duplicates().values}

        self.transform_3d = transform_3d

    def __len__(self):
        return len(self.subjects)

    def __getitem__(self, index):
        curr = self.subjects.iloc[index]
        study_path = os.path.join(self.base_path, str(curr["study_id"]))

        study_images = read_study_as_voxel_grid(study_path, self.series_descs)

        if self.transform_3d is not None:
            study_images = self.transform_3d(torch.FloatTensor(study_images))  # .data
            return study_images.to(torch.half), str(curr["study_id"])

        return torch.HalfTensor(study_images), str(curr["study_id"])


transform_3d = tio.Compose([
    tio.RescaleIntensity([0, 1]),
])

def create_subject_level_testset_and_loader(df: pd.DataFrame,
                                             transform_3d,
                                             base_path: str,
                                             batch_size=1,
                                             num_workers=0):
    testset = PatientLevelTestset(base_path, df, transform_3d=transform_3d)
    test_loader = DataLoader(testset, batch_size=batch_size, shuffle=False, num_workers=num_workers)

    return testset, test_loader

import os

data = retrieve_test_data(data_path)
dataset, dataloader = create_subject_level_testset_and_loader(data, transform_3d, os.path.join(data_path, "test_images"))

import matplotlib.pyplot as plt
import glob 
import torch


import torch
device = torch.device("cuda") if torch.cuda.is_available() else "cpu"

import torch.nn as nn
import timm_3d
from spacecutter import *
from spacecutter.losses import *
from spacecutter.models import *
from spacecutter.callbacks import *


class CNN_Model_3D_Multihead(nn.Module):
    def __init__(self,
                 backbone="efficientnet_lite0",
                 in_chans=1,
                 out_classes=5,
                 cutpoint_margin=0.15,
                 pretrained=False):
        super(CNN_Model_3D_Multihead, self).__init__()
        self.out_classes = out_classes

        self.encoder = timm_3d.create_model(
            backbone,
            features_only=False,
            drop_rate=0,
            drop_path_rate=0,
            pretrained=pretrained,
            in_chans=in_chans,
            global_pool="max"
        )
        if "efficientnet" in backbone:
            head_in_dim = self.encoder.classifier.in_features
            self.encoder.classifier = nn.Sequential(
                nn.LayerNorm(head_in_dim),
                nn.Dropout(0),
            )

        elif "vit" in backbone:
            self.encoder.head.drop = nn.Dropout(0)
            head_in_dim = self.encoder.head.fc.in_features
            self.encoder.head.fc = nn.Identity()

        self.heads = nn.ModuleList(
            [nn.Sequential(
                nn.Linear(head_in_dim, 1),
                LogisticCumulativeLink(3)
            ) for i in range(out_classes)]
        )

        self.ascension_callback = AscensionCallback(margin=cutpoint_margin)

    def forward(self, x):
        feat = self.encoder(x)
        return torch.swapaxes(torch.stack([head(feat) for head in self.heads]), 0, 1)

    def _ascension_callback(self):
        for head in self.heads:
            self.ascension_callback.clip(head[-1])

model = CNN_Model_3D_Multihead(backbone="maxvit_rmlp_tiny_rw_256", in_chans=3, out_classes=25).to(device)
model.load_state_dict(torch.load("/kaggle/input/rsna-2024/pytorch/vit_voxel_v2/6/maxvit_rmlp_tiny_rw_256_256_v2_fold_1_34.pt"))

model2 = CNN_Model_3D_Multihead(backbone="maxvit_rmlp_tiny_rw_256", in_chans=3, out_classes=25).to(device)
model2.load_state_dict(torch.load("/kaggle/input/rsna-2024/pytorch/vit_voxel_v2/6/maxvit_rmlp_tiny_rw_256_256_v2_fold_3_32.pt"))

model3 = CNN_Model_3D_Multihead(backbone="maxvit_rmlp_tiny_rw_256", in_chans=3, out_classes=25).to(device)
st = torch.load("/kaggle/input/voxel-model/maxvit_rmlp_tiny_rw_256_fold_0_35.pt")
st2 = {}
for key in st:
    st2[key.replace("backbone", "encoder")] = st[key]
model3.load_state_dict(st2)

model4 = CNN_Model_3D_Multihead(backbone="maxvit_rmlp_tiny_rw_256", in_chans=3, out_classes=25).to(device)
st = torch.load("/kaggle/input/voxel-model/3dv6/fold0/maxvit_rmlp_tiny_rw_256_fold_0_38.pt")
st2 = {}
for key in st:
    st2[key.replace("backbone", "encoder")] = st[key]
model4.load_state_dict(st2)

model5 = CNN_Model_3D_Multihead(backbone="maxvit_rmlp_tiny_rw_256", in_chans=3, out_classes=25).to(device)
st = torch.load("/kaggle/input/voxel-model/3dv6/fold1/maxvit_rmlp_tiny_rw_256_fold_1_42.pt")
st2 = {}
for key in st:
    st2[key.replace("backbone", "encoder")] = st[key]
model5.load_state_dict(st2)

model6 = CNN_Model_3D_Multihead(backbone="maxvit_rmlp_tiny_rw_256", in_chans=3, out_classes=25).to(device)
st = torch.load("/kaggle/input/voxel-model/3dv6/fold2/maxvit_rmlp_tiny_rw_256_fold_2_42.pt")
st2 = {}
for key in st:
    st2[key.replace("backbone", "encoder")] = st[key]
model6.load_state_dict(st2)

CONDITIONS = {
    "Sagittal T2/STIR": ["spinal_canal_stenosis"],
    "Axial T2": ["left_subarticular_stenosis", "right_subarticular_stenosis"],
    "Sagittal T1": ["left_neural_foraminal_narrowing", "right_neural_foraminal_narrowing"],
}

ALL_CONDITIONS = sorted(["spinal_canal_stenosis", "left_subarticular_stenosis", "right_subarticular_stenosis", "left_neural_foraminal_narrowing", "right_neural_foraminal_narrowing"])
LEVELS = ["l1_l2", "l2_l3", "l3_l4", "l4_l5", "l5_s1"]

results_df = pd.DataFrame({"row_id":[], "normal_mild": [], "moderate": [], "severe": []})

ALL_CONDITIONS

# Pre-populate results df
import glob
import os

study_ids = glob.glob("/kaggle/input/rsna-2024-lumbar-spine-degenerative-classification/test_images/*")
study_ids = [os.path.basename(e) for e in study_ids]

results_df = pd.DataFrame({"row_id":[], "normal_mild": [], "moderate": [], "severe": []})
for study_id in study_ids:
    for condition in ALL_CONDITIONS:
        for level in LEVELS:
            row_id = f"{study_id}_{condition}_{level}"
            results_df = results_df._append({"row_id": row_id, "normal_mild": 1/3, "moderate": 1/3, "severe": 1/3}, ignore_index=True)

import torch
from torch.cuda.amp import autocast
import time

start_time = time.time()

with torch.no_grad():
    with autocast(dtype=torch.float16):
        model.eval()
        model2.eval()
        model3.eval()
        model4.eval()
        model5.eval()
        model6.eval()

        for images, study_id in dataloader:
            images = images.to(device)
            print(images.size())
            _, _, s1,s2,s3 = images.size()
            images = images[:, :, (s1//2-128):(s1//2+128), (s1//2-128):(s1//2+128), (s1//2-128):(s1//2+128)]
            output = model(images)
            output2 = model2(images)
            output3 = model3(images)
            output4 = model4(images)
            output5 = model5(images)
            output6 = model6(images)
            for i, batch_out in enumerate(output):
                q1 = output.cpu().numpy()[i]
                q2 = output2.cpu().numpy()[i]
                q3 = output3.cpu().numpy()[i]
                q4 = output4.cpu().numpy()[i]
                q5 = output5.cpu().numpy()[i]
                q6 = output6.cpu().numpy()[i]
                for index, (p1, p2, p3, p4, p5, p6) in enumerate(zip(q1,q2,q3,q4,q5,q6)):
                    row_id = f"{study_id[i]}_{ALL_CONDITIONS[index // 5]}_{LEVELS[index % 5]}"
                    results_df.loc[results_df.row_id == row_id,'normal_mild'] = (p1[0]+p2[0]+p3[0]+p4[0]+p5[0]+p6[0])/6.0 #(level[0]+level2[0]+level3[0])/3
                    results_df.loc[results_df.row_id == row_id,'moderate'] = (p1[1]+p2[1]+p3[1]+p4[1]+p5[1]+p6[1])/6.0#(level[1]+level2[1]+level3[1])/3
                    results_df.loc[results_df.row_id == row_id,'severe'] = (p1[2]+p2[2]+p3[2]+p4[2]+p5[2]+p6[2])/6.0#(level[2]+level2[2]+level3[2])/3
                
print("--- %s seconds ---" % (time.time() - start_time))

results_df.to_csv('maxvit.csv', index=False)

Writing maxvit3d.py


In [14]:
import glob
import pandas as pd
subs = glob.glob('/kaggle/working/pred*.csv')
print(subs)
preds = []
for f in subs:
    df = pd.read_csv(f)
    preds.append(df)

preds = pd.concat(preds, ignore_index=True)

preds = preds.groupby(['row_id']).agg({'normal_mild':"mean", 'moderate':"mean", 'severe':"mean"}).sort_index().reset_index()
preds.to_csv('submission.csv', index=False)

['/kaggle/working/pred7.csv', '/kaggle/working/pred2.csv', '/kaggle/working/pred8.csv', '/kaggle/working/pred5.csv', '/kaggle/working/pred6.csv', '/kaggle/working/pred3.csv', '/kaggle/working/pred4.csv', '/kaggle/working/pred1.csv']


### voxel infer